# stoptimes end to end

Walks the whole chain through the Python API instead of the CLI:
read observations -> derive arrivals -> build a training frame -> run an experiment -> list/select a model -> predict -> replay-score.

No network and no committed data: a small synthetic agency (one route, four stops, four trips) is built below, the same shape `tests/e2e/test_end_to_end.py` uses so this notebook exercises the same path the CI gate does.


## 1. Build a synthetic feed

One route, four stops, four trips a day, plus a matching vehicle-position archive.

In [ ]:
import json
import os
import tempfile
import zipfile
from pathlib import Path

import pandas as pd

SOURCE = "demo"
SERVICE_DAY = "20260917"

tmp_path = Path(tempfile.mkdtemp(prefix="stoptimes-e2e-"))
os.environ["STOPTIMES_HOME"] = str(tmp_path / "home")
os.environ.pop("STOPTIMES_MODEL_REGISTRY_DIR", None)
os.environ.pop("MODEL_REGISTRY_DIR", None)

from stoptimes.core.config import reset_config
reset_config()


In [ ]:
def build_static_zip(path: Path) -> Path:
    """A one-route, four-stop line running west to east, four trips a day."""
    stops = ["stop_id,stop_name,stop_lat,stop_lon"]
    for i in range(4):
        stops.append(f"s{i + 1},Stop {i + 1},0.0,{i * 0.01:.5f}")

    trips = ["route_id,service_id,trip_id,direction_id,shape_id"]
    stop_times = ["trip_id,arrival_time,departure_time,stop_id,stop_sequence,shape_dist_traveled"]
    for t in range(4):
        trip_id = f"t{t + 1}"
        trips.append(f"r1,svc,{trip_id},0,sh1")
        for i in range(4):
            clock = f"{8 + t:02d}:{i * 2:02d}:00"
            stop_times.append(f"{trip_id},{clock},{clock},s{i + 1},{i + 1},{i * 1113}")

    shapes = ["shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled"]
    for i in range(4):
        shapes.append(f"sh1,0.0,{i * 0.01:.5f},{i + 1},{i * 1113}")

    with zipfile.ZipFile(path, "w") as zf:
        zf.writestr("stops.txt", "\n".join(stops) + "\n")
        zf.writestr("trips.txt", "\n".join(trips) + "\n")
        zf.writestr("stop_times.txt", "\n".join(stop_times) + "\n")
        zf.writestr("shapes.txt", "\n".join(shapes) + "\n")
        zf.writestr("routes.txt", "route_id,route_short_name,route_long_name\nr1,1,Line One\n")
    return path


def build_archive(root: Path) -> Path:
    """Vehicle positions for those trips, in the layout the recorder writes."""
    rows = []
    for t in range(4):
        start = pd.Timestamp(f"2026-09-17T{8 + t:02d}:00:00Z")
        for step in range(25):  # every 15 s, the whole line
            rows.append(
                {
                    "source_id": SOURCE,
                    "kind": "vehicle_position",
                    "trip_id": f"t{t + 1}",
                    "route_id": "r1",
                    "vehicle_id": f"v{t + 1}",
                    "start_date": SERVICE_DAY,
                    "ts": start + pd.Timedelta(seconds=15 * step),
                    "lat": 0.0,
                    "lon": round(step * 0.00125, 6),
                    "bearing": 90.0,
                }
            )
    partition = root / "year=2026" / "month=09" / "day=17" / "route_id=r1"
    partition.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_parquet(partition / "part-0.parquet")
    return root


def build_agency_dir(tmp_path: Path) -> Path:
    directory = tmp_path / "agencies"
    directory.mkdir(exist_ok=True)
    (directory / f"{SOURCE}.yaml").write_text(
        f"name: {SOURCE}\n"
        "gtfs_static_url: unused.zip\n"
        "vehicle_positions_url: http://example.invalid/vp.json\n"
        "trip_updates_url: http://example.invalid/tu.json\n"
        "poll_interval_s: 15\n"
        "timezone: UTC\n"
        "arrival_radius_m: 60.0\n"
        "dwell_s: 0.0\n"
        "shape_threshold_m: 50.0\n",
        encoding="utf-8",
    )
    return directory


static = build_static_zip(tmp_path / "gtfs.zip")
archive = build_archive(tmp_path / "archive")
agencies = build_agency_dir(tmp_path)
registry = tmp_path / "registry"
static, archive, agencies, registry


## 2. Register the recorded positions in the local catalog

In [ ]:
from stoptimes.core.api import add_dataset, list_datasets

entry = add_dataset("positions", str(archive), source=SOURCE, kind="observations")
print(entry.name, entry.kind.value, entry.row_count, "rows")
list_datasets()


## 3. Positions -> actual arrivals (the ground truth)

In [ ]:
from stoptimes.core.api import build_arrivals_dataset

arrivals_frame, arrivals_summary = build_arrivals_dataset(
    observations="positions",
    static=str(static),
    source=SOURCE,
    sources_dir=str(agencies),
    out=str(tmp_path / "arrivals.parquet"),
    register="arrivals",
)
print(arrivals_summary)
arrivals_frame.head()


## 4. Arrivals + positions -> training frame

In [ ]:
from stoptimes.train.api import build_frame

frame, frame_summary = build_frame(
    observations="positions",
    arrivals="arrivals",
    static=str(static),
    source=SOURCE,
    sources_dir=str(agencies),
    out=str(tmp_path / "frame.parquet"),
    register="frame",
)
print(frame_summary)
frame.head()


## 5. Train a model from an experiment

In [ ]:
experiment_yaml = tmp_path / "experiment.yaml"
experiment_yaml.write_text(
    f"""name: e2e-historical-mean
data:
  datasets: [frame]
target: arrival_delay_s
model:
  family: historical_mean
  params: {{}}
features:
  - distance_to_stop
  - stops_ahead
  - hour
split:
  kind: time
  train_fraction: 0.7
  test_fraction: 0.3
seed: 7
outputs:
  registry_dir: {registry}
  tags: [e2e]
baselines: []
""",
    encoding="utf-8",
)

from stoptimes.train.api import plan_experiment, load_experiment, run_experiment_from_file

print(json.dumps(plan_experiment(load_experiment(experiment_yaml)), indent=2))


In [ ]:
meta = run_experiment_from_file(experiment_yaml)
print(meta.artifact_id, meta.family, meta.metrics.get("test_mae_seconds"))


## 6. List and select a model

In [ ]:
from stoptimes.core.api import list_models, model_card, resolve_source

rows = list_models(registry=registry)
rows


In [ ]:
from stoptimes.train.api import select_model

src = resolve_source(SOURCE, sources_dir=agencies)
picked = select_model(registry=registry, source_id=SOURCE, artifact_id=meta.artifact_id)
picked.artifact_id, picked.metrics


## 7. Predict: run one serving pipeline pass

A tiny user pipeline overrides `ingest`/`features` to read a fixture TripUpdates feed instead of polling a live URL -- the customizable-inference story from the product page.

In [ ]:
def build_trip_updates_feed(path: Path) -> Path:
    entities = []
    for t in range(4):
        stop_time_updates = [
            {
                "stop_sequence": i + 1,
                "stop_id": f"s{i + 1}",
                "arrival": {"time": 1789639200 + t * 3600 + i * 120},
            }
            for i in range(4)
        ]
        entities.append(
            {
                "id": f"e{t + 1}",
                "trip_update": {
                    "trip": {
                        "trip_id": f"t{t + 1}",
                        "route_id": "r1",
                        "direction_id": 0,
                        "start_date": SERVICE_DAY,
                    },
                    "vehicle": {"id": f"v{t + 1}"},
                    "stop_time_update": stop_time_updates,
                    "timestamp": 1789639200 + t * 3600,
                },
            }
        )
    path.write_text(
        json.dumps({"header": {"gtfs_realtime_version": "2.0", "timestamp": 1789639200}, "entity": entities}),
        encoding="utf-8",
    )
    return path


feed_path = build_trip_updates_feed(tmp_path / "incoming_tu.json")

PIPELINE_SRC = '''
import json
from pathlib import Path

import pandas as pd
from stoptimes.core.contracts.data import Observation, ObservationKind
from stoptimes.serve.pipeline import build_default_pipeline, default_features

pipeline = build_default_pipeline()


@pipeline.stage("ingest")
def ingest_from_file(_payload, ctx):
    feed = json.loads(Path("__FEED_PATH__").read_text(encoding="utf-8"))
    observations = []
    for entity in feed["entity"]:
        tu = entity["trip_update"]
        trip = tu["trip"]
        for stu in tu["stop_time_update"]:
            observations.append(
                Observation(
                    kind=ObservationKind.TRIP_UPDATE,
                    source_id=ctx.source.id,
                    trip_id=trip["trip_id"],
                    route_id=trip.get("route_id"),
                    direction_id=trip.get("direction_id"),
                    start_date=trip.get("start_date"),
                    vehicle_id=(tu.get("vehicle") or {}).get("id"),
                    ts=pd.Timestamp(tu["timestamp"], unit="s", tz="UTC"),
                    stop_id=stu["stop_id"],
                    stop_sequence=stu["stop_sequence"],
                    arrival_time=pd.Timestamp(stu["arrival"]["time"], unit="s", tz="UTC"),
                )
            )
    return observations


@pipeline.stage("features")
def features_with_distance(matched, ctx):
    frame = default_features(matched, ctx)
    # the trained model wants these; a real deployment gets them from the
    # match stage's map-matching against the static feed
    frame["distance_to_stop"] = 300.0
    frame["stops_ahead"] = frame["stop_sequence"].astype(float)
    return frame
'''

pipeline_file = tmp_path / "my_pipeline.py"
pipeline_file.write_text(
    PIPELINE_SRC.replace("__FEED_PATH__", str(feed_path)), encoding="utf-8"
)


In [ ]:
from stoptimes.train.api import predict_once

served = tmp_path / "trip_updates.json"
summary = predict_once(
    source=src,
    registry=registry,
    artifact_id=meta.artifact_id,
    pipeline=f"{pipeline_file}:pipeline",
    sink=f"file:{served}",
    format="json",
)
summary


## 8. Replay-score: validate that trained model against the recorded day

In [ ]:
from stoptimes.val.api_verbs import replay_score, score_report

scored = replay_score(
    source=SOURCE,
    archive=str(archive),
    static=str(static),
    sources_dir=str(agencies),
    start="2026-09-17",
    end="2026-09-18",
    predictor_ids=["baseline:schedule"],
)
print(len(scored), "scored predictions")
scored.head()


In [ ]:
report = score_report(scored)
report.per_predictor
